# 03 - Content-Based Filtering

In [2]:
import sys
import os
os.chdir('/Users/mariaparis/Downloads/recommender_assignment_placeholders')
sys.path.insert(0, '/Users/mariaparis/Downloads/recommender_assignment_placeholders')

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import config
from src.data_loading import load_ratings, load_items, train_test_split_ratings
from src.content_based import ContentBasedRecommender

In [4]:
ratings = load_ratings()
items   = load_items()
train, test = train_test_split_ratings(ratings, test_size=0.2)
title_map = items.set_index(config.ITEM_COL)[config.TITLE_COL].to_dict()
genre_map = items.set_index(config.ITEM_COL)[config.GENRES_COL].to_dict()

## 1. How it works

The pipeline has three steps:

1. **Item vectors**: each movie is represented as a TF-IDF vector built from its genres. TF-IDF down-weights very common genres like Drama so they do not dominate every recommendation.
2. **User profile**: a weighted sum of the item vectors for movies the user has rated, where the weight is the rating centered on the user's mean. Liked movies (above average) pull the profile towards their genre space; disliked ones push it away.
3. **Scoring**: we compute cosine similarity between the user profile and every candidate item, then return the top-N.

## 2. Fit the model

In [5]:
cb = ContentBasedRecommender()
cb.fit(train, items)

print(f"Item feature matrix shape: {cb.item_features_.shape}")
print(f"Number of genre tokens    : {cb.item_features_.shape[1]}")

Item feature matrix shape: (9742, 24)
Number of genre tokens    : 24


## 3. User profile inspection

Let's look at what User 1 has rated highly and check whether the resulting profile makes sense.

In [6]:
uid = 1
user_ratings = train[train[config.USER_COL] == uid].merge(items, on=config.ITEM_COL)
top_rated = user_ratings.sort_values(config.RATING_COL, ascending=False).head(10)
top_rated[[config.TITLE_COL, config.GENRES_COL, config.RATING_COL]]

,title,genres,rating
0,"Rescuers, The (1977)",Adventure|Animation|Children|Crime|Drama,5.0
91,"Messenger: The Story of Joan of Arc, The (1999)",Drama|War,5.0
114,"Terminator, The (1984)",Action|Sci-Fi|Thriller,5.0
180,Crocodile Dundee (1986),Adventure|Comedy,5.0
111,Desperado (1995),Action|Romance|Western,5.0
109,"Dirty Dozen, The (1967)",Action|Drama|War,5.0
181,"American Tail, An (1986)",Adventure|Animation|Children|Comedy,5.0
106,"Goonies, The (1985)",Action|Adventure|Children|Comedy|Fantasy,5.0
103,"Fugitive, The (1993)",Thriller,5.0
102,"Iron Giant, The (1999)",Adventure|Animation|Children|Drama|Sci-Fi,5.0


In [7]:
profile = cb.build_user_profile(uid, train)
feature_names = cb.vectorizer.get_feature_names_out()

top_features = sorted(zip(feature_names, profile), key=lambda x: x[1], reverse=True)[:10]
print(f"Top genres in User {uid}'s profile:")
for genre, weight in top_features:
    print(f"  {genre:<20} {weight:.4f}")

Top genres in User 1's profile:
  drama                0.0281
  war                  0.0272
  animation            0.0222
  musical              0.0200
  children             0.0193
  adventure            0.0170
  romance              0.0003
  documentary          0.0000
  film                 0.0000
  genres               0.0000


The profile weights reflect the genres this user has rated above their personal average. This is what gets compared against all items to produce recommendations.

## 4. Recommendations for three users

In [8]:
for uid in [1, 50, 100]:
    recs = cb.recommend(uid, train, n=5)
    print(f"\nUser {uid}:")
    for rank, iid in enumerate(recs, 1):
        print(f"  {rank}. {title_map.get(iid, iid)} — {genre_map.get(iid, '')}")


User 1:
  1. Anastasia (1997) — Adventure|Animation|Children|Drama|Musical
  2. Grave of the Fireflies (Hotaru no haka) (1988) — Animation|Drama|War
  3. Dumbo (1941) — Animation|Children|Drama|Musical
  4. Tarzan (1999) — Adventure|Animation|Children|Drama
  5. Up (2009) — Adventure|Animation|Children|Drama

User 50:
  1. Nixon (1995) — Drama
  2. Othello (1995) — Drama
  3. Dangerous Minds (1995) — Drama
  4. Cry, the Beloved Country (1995) — Drama
  5. Restoration (1995) — Drama

User 100:
  1. Leaving Las Vegas (1995) — Drama|Romance
  2. How to Make an American Quilt (1995) — Drama|Romance
  3. When Night Is Falling (1995) — Drama|Romance
  4. Once Upon a Time... When We Were Colored (1995) — Drama|Romance
  5. Angels and Insects (1995) — Drama|Romance


## 5. Similar items

A useful side feature of content-based models: finding movies similar to a given one based purely on genre overlap.

In [9]:
# Movies similar to Toy Story (movieId = 1)
similar = cb.similar_items(item_id=1, n=8)
print("Movies similar to Toy Story (1995):")
for iid in similar:
    print(f"  {title_map.get(iid, iid)} — {genre_map.get(iid, '')}")

Movies similar to Toy Story (1995):
  Antz (1998) — Adventure|Animation|Children|Comedy|Fantasy
  Toy Story 2 (1999) — Adventure|Animation|Children|Comedy|Fantasy
  Adventures of Rocky and Bullwinkle, The (2000) — Adventure|Animation|Children|Comedy|Fantasy
  Emperor's New Groove, The (2000) — Adventure|Animation|Children|Comedy|Fantasy
  Monsters, Inc. (2001) — Adventure|Animation|Children|Comedy|Fantasy
  Wild, The (2006) — Adventure|Animation|Children|Comedy|Fantasy
  Shrek the Third (2007) — Adventure|Animation|Children|Comedy|Fantasy
  Tale of Despereaux, The (2008) — Adventure|Animation|Children|Comedy|Fantasy


## 6. Strengths and limitations

**Strengths:**
- Works for new items with no rating history (no item cold-start)
- Transparent: recommendations can be explained by the genres the user has liked
- Does not require data from other users

**Limitations:**
- Creates a filter bubble: the model only recommends more of what the user already knows they like
- Limited by the quality and granularity of item metadata; genres are a coarse signal
- Cannot discover surprising or serendipitous recommendations across genre boundaries